# 12 - Statistical and interpretability analysis

This notebook reproduces:

* McNemar tests against the classifier baseline.
* Paired bootstrap confidence intervals for precision, recall and F1.
* Gain/loss analysis inside classifier-probability bins.
* Positive Correction Rate (PCR) for the classic and light prompts.

It reads the result files produced by notebooks 08-11.

## Setup

In [ ]:
import numpy as np
import pandas as pd

from spam_detection.config import RESULTS_DIR, TAU_HIGH, TAU_LOW
from spam_detection.evaluation import predictions_from_results
from spam_detection.statistics import (
    bin_gain_analysis,
    bootstrap_confidence_test,
    mcnemar_test,
    plot_bootstrap_results,
    positive_correction_rates,
    summarize_table,
)

TABLES_DIR = RESULTS_DIR / "tables" / "statistical_analysis"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Load the generated runs

In [ ]:
SCENARIOS = {
    "In-Domain": RESULTS_DIR / "runs" / "in_domain",
    "Hotel Reviews (zero-shot)": RESULTS_DIR / "runs" / "hotel_reviews_zero_shot",
    "Hotel Reviews (adapted)": RESULTS_DIR / "runs" / "hotel_reviews_adapted",
    "Cross-Domain (zero-shot)": RESULTS_DIR / "runs" / "cross_domain_zero_shot",
    "Cross-Domain (adapted)": RESULTS_DIR / "runs" / "cross_domain_adapted",
}


def load_run(directory, name):
    path = directory / name
    return pd.read_csv(path) if path.exists() else None


runs = {}
for scenario, directory in SCENARIOS.items():
    runs[scenario] = {
        "clf": load_run(directory, "clf_only.csv"),
        "interp": load_run(directory, "interpolation.csv"),
        "rag": load_run(directory, "full_rag.csv"),
        "light": load_run(directory, "full_rag_light.csv"),
    }
    available = [key for key, value in runs[scenario].items() if value is not None]
    print(f"{scenario}: {available}")

## McNemar comparison tables

In [ ]:
for scenario, data in runs.items():
    if data["clf"] is None:
        continue
    y_true = data["clf"]["true_label"].astype(int).to_numpy()
    predictions = {"RoBERTa-LoRA": predictions_from_results(data["clf"])}
    if data["interp"] is not None:
        predictions["Interpolation"] = predictions_from_results(data["interp"])
    if data["rag"] is not None:
        predictions["Full RAG"] = predictions_from_results(data["rag"])
    if data["light"] is not None:
        predictions["Full RAG (light prompt)"] = predictions_from_results(data["light"])

    table = summarize_table(
        dataset_name=scenario,
        y_true=y_true,
        preds=predictions,
        baseline="RoBERTa-LoRA",
        mcnemar_fn=mcnemar_test,
        eps_mcnemar=True,
    )
    display(table)
    safe_name = scenario.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")
    table.to_csv(TABLES_DIR / f"mcnemar_{safe_name}.csv", index=False)

## Paired bootstrap confidence intervals

In [ ]:
for scenario, data in runs.items():
    if data["clf"] is None or data["rag"] is None:
        continue
    y_true = data["clf"]["true_label"].astype(int).to_numpy()
    baseline = predictions_from_results(data["clf"])
    rag = predictions_from_results(data["rag"])

    bootstrap = bootstrap_confidence_test(y_true, baseline, rag, n_bootstraps=2000, seed=42)
    display(bootstrap)

    safe_name = scenario.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")
    bootstrap.to_csv(TABLES_DIR / f"bootstrap_{safe_name}.csv")
    plot_bootstrap_results(bootstrap, title=f"Bootstrap CI - {scenario}")

## Gain/loss inside classifier-probability bins

In [ ]:
for scenario, data in runs.items():
    if data["clf"] is None or data["rag"] is None:
        continue
    y_true = data["clf"]["true_label"].astype(int).to_numpy()
    baseline = predictions_from_results(data["clf"])
    rag = predictions_from_results(data["rag"])

    bins = bin_gain_analysis(
        y=y_true,
        p=data["clf"]["p_clf"].to_numpy(),
        pred_base=baseline,
        pred_rag=rag,
        name_base="RoBERTa-LoRA",
        name_rag="Full RAG",
        graph_title=f"Gain / loss - {scenario}",
        plot=True,
    )
    display(bins)
    safe_name = scenario.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")
    bins.to_csv(TABLES_DIR / f"bin_gain_{safe_name}.csv", index=False)

## Positive Correction Rate and grounding metrics

In [ ]:
grounding = {
    "In-Domain": RESULTS_DIR / "tables" / "interpretability" / "id_interp.csv",
    "Hotel Reviews (adapted)": RESULTS_DIR / "tables" / "interpretability" / "hrft_interp.csv",
    "Cross-Domain (adapted)": RESULTS_DIR / "tables" / "interpretability" / "cdft_interp.csv",
}
scenario_keys = {
    "In-Domain": "In-Domain",
    "Hotel Reviews (adapted)": "Hotel Reviews (adapted)",
    "Cross-Domain (adapted)": "Cross-Domain (adapted)",
}

rows = []
for label, path in grounding.items():
    data = runs.get(scenario_keys[label], {})
    if data.get("rag") is None or data.get("light") is None:
        continue
    classic_pcr, light_pcr = positive_correction_rates(data["rag"], data["light"], TAU_LOW, TAU_HIGH)
    frame = pd.read_csv(path) if path.exists() else pd.DataFrame()
    rows.append(
        {
            "Scenario": label,
            "Classic PCR (%)": round(100 * classic_pcr, 1),
            "Light PCR (%)": round(100 * light_pcr, 1),
            "EGR classic (%)": float(frame.loc[frame["Prompt Setup"].str.contains("RAG$", regex=True), "Evidence Grounding Rate (%)"].iloc[0]) if not frame.empty else np.nan,
            "EGR light (%)": float(frame.loc[frame["Prompt Setup"].str.contains("light"), "Evidence Grounding Rate (%)"].iloc[0]) if not frame.empty else np.nan,
        }
    )

pcr_table = pd.DataFrame(rows)
display(pcr_table)
pcr_table.to_csv(TABLES_DIR / "interpretability_pcr.csv", index=False)